In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
from scipy.constants import g
from uncertainties import ufloat
import uncertainties.umath as unm
from uncertainties import unumpy as unp
from uncertainties import ufloat
import sys
import os
percorso_main = os.path.abspath('..')
if percorso_main not in sys.path:
    sys.path.append(percorso_main)
from Libreria_python import analisi_dati as ad
from scipy.optimize import curve_fit

# Raccolta Masse

In [3]:
df=pd.read_excel('Urti.xlsx', sheet_name=0)
df=df.fillna('')
df

,Unnamed: 0,Carrello rosso (kg),Carrello blu (kg),massa 1 (kg),massa 2 (kg),Massa totale (kg),massa Libro 1 (kg),massa Libro 2 (kg),Errore sulla massa (kg)
0,Massa,,0.25452,0.25315,0.25375,0.76142,0.29007,0.18594,0.0005
1,Massa con magnete,0.25634,0.27538,,,0.78228,,,
2,Massa con molla,,0.25687,,,0.76324,,,


In [4]:
m_carrello=ufloat(df.iloc[0, 2], df.iloc[0, 8])
m_carrello_masse=ufloat(df.iloc[0, 5], df.iloc[0, 8])

# Seconda Legge di Newton: Piano Orizzontale


In [5]:
df1=pd.read_excel('Urti.xlsx', sheet_name=1)
df1=df1.fillna('')
df1

,Unnamed: 0,Interpolazione carrello,Unnamed: 2,Interpolazione carrello+masse,Unnamed: 4
0,,m (kg),sm (kg),m (kg),sm (kg)
1,1,0.249,0.04,0.769,0.043
2,2,0.253,0.043,0.755,0.038
3,3,0.247,0.042,0.749,0.038
4,x_best,0.249594,,0.756774,
5,sx_best,0.024023,,0.022787,


In [6]:
m1_interpolata=ufloat(df1.iloc[4, 1], df1.iloc[5, 1])
m2_interpolata=ufloat(df1.iloc[4, 3], df1.iloc[5, 3])

# Seconda Legge di Newton: Piano Inclinato

In [9]:
df2=pd.read_excel('Urti.xlsx', sheet_name=2)
df2=df2.fillna('')
df2

,Unnamed: 0,Interpolazione piano inclinato,Unnamed: 2,Unnamed: 3,Unnamed: 4,Angolo,incertezza
0,,m,sm,θ,sθ,10.0,1.0
1,1,0.264,0.002,0.176,0.002,,
2,2,0.257,0.002,0.16,0.003,,
3,3,0.26,0.003,0.175,0.0023,,
4,x_best,0.260409,,0.172425,,,
5,sx_best,0.001279,,0.001348,,,


In [10]:
m3_interpolata=ufloat(df2.iloc[4, 1], df2.iloc[5, 1])
theta_misurato=ufloat(np.radians(df2.iloc[0, 5]), np.radians(df2.iloc[0, 6]))
theta_interpolato=ufloat(df2.iloc[4, 3], df2.iloc[5, 3])

In [11]:
print("=" * 80)
ad.t_test(m1_interpolata, m_carrello, nome1="m1_interpolata", nome2="m_carrello");
ad.t_test(m2_interpolata, m_carrello_masse, nome1="m2_interpolata", nome2="m_carrello_masse");
ad.t_test(m3_interpolata, m_carrello, nome1="m3_interpolata", nome2="m_carrello");
ad.t_test(theta_interpolato, theta_misurato, nome1="theta_interpolato", nome2="theta_misurato");
print("=" * 80)

Test di compatibilità tra m1_interpolata e m_carrello: |t| = 0.20499809
Test di compatibilità tra m2_interpolata e m_carrello_masse: |t| = 0.20383887
Test di compatibilità tra m3_interpolata e m_carrello: |t| = 4.2878102
Test di compatibilità tra theta_interpolato e theta_misurato: |t| = 0.12042043


# Teorema dell'Impulso


In [12]:
df3=pd.read_excel('Urti.xlsx', sheet_name=3)
df3=df3.fillna('')
df3

,Unnamed: 0,Massa (kg),Area (N*s),Incertezza (N*s) (semidisp.),Velocità iniziale (m/s),Incertezza (dev.standard),Velocità finale (m/s),Incertezza (dev.standard).1
0,Magnete,0.27538,-0.261,0.02,0.558,0.008,-0.541,0.007
1,Molla,0.25687,-0.286,0.04,0.640,0.007,-0.624,0.006


In [13]:
i_magnete_sperimentale = ufloat(df3.iloc[0, 2], df3.iloc[0, 3].round(3))
i_molla_sperimentale = ufloat(df3.iloc[1, 2], df3.iloc[1, 3].round(3))
m_magnete = ufloat(df3.iloc[0, 1], 0.0005)
m_molla = ufloat(df3.iloc[1, 1], 0.0005)
vi_magnete = ufloat(df3.iloc[0, 4], df3.iloc[0, 5].round(3))
vi_molla = ufloat(df3.iloc[1, 4], df3.iloc[1, 5].round(3))
vf_magnete = ufloat(df3.iloc[0, 6], df3.iloc[0, 7].round(3))
vf_molla = ufloat(df3.iloc[1, 6], df3.iloc[1, 7].round(3))

i_magnete_teorico = m_molla * (vf_magnete - vi_magnete)
i_molla_teorico = m_molla * (vf_molla - vi_molla)

d_E_magnete = 0.5 * m_magnete * (vf_magnete**2 - vi_magnete**2)
d_E_molla = 0.5 * m_molla * (vf_molla**2 - vi_molla**2)

print("=" * 80)
print(f"Impulso exp magnete  = {i_magnete_sperimentale:.3P}")
print(f"Impulso teo magnete  = {i_magnete_teorico:.3P}")
print(f"Impulso exp molla  = {i_molla_sperimentale:.3P}")
print(f"Impulso teo molla  = {i_molla_teorico:.3P}")
print("=" * 80)
ad.t_test(i_magnete_sperimentale, i_magnete_teorico, "I_exp_magnete", "I_teo_magnete");
ad.t_test(i_molla_sperimentale, i_molla_teorico, "I_exp_molla", "I_teo_molla");
print("=" * 80)
print(f"Variazione energia cinetica magnete  = {d_E_magnete:.2P}")
print(f"Variazione energia cinetica molla  = {d_E_molla:.2P}")
print("=" * 80)
ad.compatibilita_valore(d_E_magnete, 0, "dE_magnete")
ad.compatibilita_valore(d_E_molla, 0, "dE_molla")
print("=" * 80)

Impulso exp magnete  = -0.261±0.020
Impulso teo magnete  = -0.282±0.003
Impulso exp molla  = -0.286±0.040
Impulso teo molla  = -0.325±0.002
Test di compatibilità tra I_exp_magnete e I_teo_magnete: |t| = 1.0548265
Test di compatibilità tra I_exp_molla e I_teo_molla: |t| = 0.96528141
Variazione energia cinetica magnete  = -0.0026±0.0016
Variazione energia cinetica molla  = -0.0026±0.0015
Compatibilità di dE_magnete con 0: |t| = 1.595755
Compatibilità di dE_molla con 0: |t| = 1.7319456


# Urti tra Carrelli

In [14]:
df4=pd.read_excel('Urti.xlsx', sheet_name=4)
df4=df4.fillna('')
df4

,Unnamed: 0,Carrello rosso,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Carrello blu,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,,Massa (kg),Velocità iniziale (m/s),sv (m/s),Velocità finale (m/s),sv (m/s),Massa (kg),Velocità iniziale (m/s),Velocità finale (m/s),sv (m/s)
1,Magnete,0.25634,0.518,0.028,-0.019,0.008,0.27538,0,0.491,0.01
2,,0.25634,0.509,0.016,-0.213,0.013,0.78228,0,0.24,0.004
3,,0.76324,0.346,0.013,0.163,0.01,0.27538,0,0.485,0.014
4,Velcro,0.25634,0.569,0.014,0.251,0.024,0.27538,0,0.251,0.024
5,,0.25634,0.569,0.004,0.132,0.012,0.78228,0,0.132,0.012
6,,0.76324,0.48,0.004,0.344,0.015,0.27538,0,0.344,0.015


In [15]:
m_R = unp.uarray(df4.iloc[1:4, 1].to_numpy(), 0.0005)
m_B = unp.uarray(df4.iloc[1:4, 6].to_numpy(), 0.0005)
v_i_RM = unp.uarray(df4.iloc[1:4, 2].to_numpy(), df4.iloc[1:4, 3].to_numpy())

vR_fM = unp.uarray(df4.iloc[1:4, 4].to_numpy(), df4.iloc[1:4, 5].to_numpy())
vR_bM = unp.uarray(df4.iloc[1:4, 8].to_numpy(), df4.iloc[1:4, 9].to_numpy())
v_iRV = unp.uarray(df4.iloc[4:7, 2].to_numpy(), df4.iloc[4:7, 3].to_numpy())
v_fV = unp.uarray(df4.iloc[4:7, 4].to_numpy(), df4.iloc[4:7, 5].to_numpy())

v_f_R = ((m_R - m_B) / (m_R + m_B)) * v_i_RM
v_f_B = ((2 * m_R) / (m_R + m_B)) * v_i_RM

print("=" * 80)

for i in range(len(m_R)):
    print(f"Set {i+1}: Velocità finale R = {v_f_R[i]:.3P}\n       Velocità finale B = {v_f_B[i]:.3P}")
    ad.t_test(vR_fM[i], v_f_R[i], "Vf R misurata", "Vf R teorica");
    ad.t_test(vR_bM[i], v_f_B[i], "Vf B misurata", "Vf B teorica");
    print("=" * 80)

v_f = (m_R / (m_R + m_B)) * v_iRV
d_E = 0.5 * (m_R + m_B) * v_f**2 - 0.5 * m_R  * v_iRV**2

for i in range(len(m_B)):
    print(f"Set {i+1}: Velocità finale = {v_f[i]:.3P}")
    ad.t_test(v_fV[i], v_f[i], "Vf misurata", "Vf teorica");
    print(f"Set {i+1}: Variazione energia cinetica = {d_E[i]:.3P} : Urto Totalmente Anelastico")
    print("=" * 80)

Set 1: Velocità finale R = -0.0185±0.0012
       Velocità finale B = 0.499±0.027
Test di compatibilità tra Vf R misurata e Vf R teorica: |t| = 0.055769938
Test di compatibilità tra Vf B misurata e Vf B teorica: |t| = 0.29346651
Set 2: Velocità finale R = -0.258±0.008
       Velocità finale B = 0.251±0.008
Test di compatibilità tra Vf R misurata e Vf R teorica: |t| = 2.9203861
Test di compatibilità tra Vf B misurata e Vf B teorica: |t| = 1.2696208
Set 3: Velocità finale R = 0.163±0.006
       Velocità finale B = 0.509±0.019
Test di compatibilità tra Vf R misurata e Vf R teorica: |t| = 0.040706551
Test di compatibilità tra Vf B misurata e Vf B teorica: |t| = 0.99303206
Set 1: Velocità finale = 0.274±0.007
Test di compatibilità tra Vf misurata e Vf teorica: |t| = 0.93497521
Set 1: Variazione energia cinetica = -0.0215±0.0011 : Urto Totalmente Anelastico
Set 2: Velocità finale = 0.140±0.001
Test di compatibilità tra Vf misurata e Vf teorica: |t| = 0.70034493
Set 2: Variazione energia cinet

# Attrito Statico


In [16]:
df5=pd.read_excel('Urti.xlsx', sheet_name=5)
df5=df5.fillna('')
df5

,1 libro,Unnamed: 1,2 libri,Unnamed: 3
0,Massa (kg),F (N),Massa (kg),F (N)
1,0.29007,0.681,0.47601,0.936


In [17]:
μ_s1=(ufloat(df5.iloc[1, 1], 0.015))/(ufloat(df5.iloc[1, 0], 0.0005)*g)
μ_s2=(ufloat(df5.iloc[1, 3], 0.015))/(ufloat(df5.iloc[1, 2], 0.0005)*g)
print("="*70)
print(f"Il valore misurato di μ_s1 è: {μ_s1:.3P}")
print(f"Il valore misurato di μ_s2 è: {μ_s2:.3P}")
ad.t_test(μ_s1, μ_s2, nome1="μ_s1", nome2="μ_s2")
print("="*70)

Il valore misurato di μ_s1 è: 0.239±0.005
Il valore misurato di μ_s2 è: 0.201±0.003
Test di compatibilità tra μ_s1 e μ_s2: |t| = 6.279997


# Attrito Dinamico

In [18]:
df6=pd.read_excel('Urti.xlsx', sheet_name=6)
df6=df6.fillna('')
df6

,Unnamed: 0,Coefficiente angolare
0,,1.478000
1,,1.519200
2,,1.511300
3,,1.517600
4,,1.524700
5,Media,1.510160
6,Errore della media,0.008319


In [19]:
a=ufloat(df6.iloc[5, 1], df6.iloc[6,1])
theta = ufloat(np.radians(10), np.radians(1))
μ_d= unp.tan(theta)-a/(g*unp.cos(theta))
μ_d_atteso = 0.005                                
print("="*70)
print(f"Il valore misurato di μ_d è: {μ_d:.2P}")
ad.compatibilita_valore(μ_d, μ_d_atteso)
print("="*70)

Il valore misurato di μ_d è: 0.020±0.018
Compatibilità di misura con 0.005: |t| = 0.85298935
